In [ ]:
# Change to project root if not already there
import os

print(f"Initial working directory: {os.getcwd()}")

# Find project root by looking for config and templates folders
# Search up to 3 levels up
found = False
for _ in range(4):
    if os.path.exists('config') and os.path.exists('templates'):
        found = True
        break
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():  # Already at root
        break
    os.chdir(parent)

if not found:
    raise FileNotFoundError(f"Cannot find project root (config and templates folders not found). Current dir: {os.getcwd()}")

print(f"Project root: {os.getcwd()}")

config_dir = "config/car_coll/v1"

In [ ]:
import os
import subprocess
import sys
import yaml

config_path = os.path.join(config_dir, 'config.yaml')

if not os.path.exists(config_path):
    raise FileNotFoundError(f"{config_path} not found")

# Load config to get template name
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

template_name = cfg['experiment'].get('template', 'standard_gbm')
template_path = f'templates/{template_name}.ipynb'

if not os.path.exists(template_path):
    raise FileNotFoundError(f"Template {template_path} not found")

# Extract model name from config_dir (e.g., 'car_coll' from 'config/car_coll/v1')
parts = config_dir.split('/')
model_name = parts[1]  # car_coll, suv_comp, etc.

output_dir = config_dir.replace('config/', 'output/', 1)
os.makedirs(output_dir, exist_ok=True)
output_notebook = os.path.join(output_dir, f'{model_name}.ipynb')

print(f"Model: {model_name}")
print(f"Template: {template_name}")
print(f"Config: {config_path}")
print(f"Output: {output_notebook}")

In [ ]:
# Run papermill in a SUBPROCESS to ensure memory isolation
# This prevents memory accumulation from executed notebooks
# Use py39_26v1 kernel to ensure correct SHAP version
# Run from templates/ folder so relative paths work
cmd = [
    sys.executable, '-m', 'papermill',
    template_path,
    output_notebook,
    '-p', 'config_path', f'../{config_path}',
    '-k', 'py39_26v1',
    '--cwd', 'templates'
]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False)

if result.returncode == 0:
    print(f"\n✓ Done: {output_notebook}")
else:
    print(f"\n✗ Failed with return code {result.returncode}")

In [ ]:
# Copy supporting files to make output self-contained
import shutil

files_to_copy = [
    # (source, dest_name)
    ('lib/gbm_functions.ipynb', 'gbm_functions.ipynb'),
    ('lib/utils.py', 'utils.py'),
    (os.path.join(config_dir, 'config.yaml'), 'config.yaml'),
    (os.path.join(config_dir, 'feature_selection.csv'), 'feature_selection.csv'),
    (os.path.join(config_dir, 'feature_clipping.csv'), 'feature_clipping.csv'),
]

print(f"\nCopying files to {output_dir}:")
for src, dest_name in files_to_copy:
    if os.path.exists(src):
        dest = os.path.join(output_dir, dest_name)
        shutil.copy2(src, dest)
        print(f"  ✓ {dest_name}")
    else:
        print(f"  ⚠ {src} not found (skipped)")

print(f"\nOutput folder is now self-contained: {output_dir}")